# 07 · Loops and the recursion limit

**Goal:** by the end you can build a graph that repeats a node until a condition is met, and protect it with the
recursion limit so a mistake can't run forever.

## The idea in plain English

A **loop** is just an edge that points back to a node that has already run. We make it with a conditional edge: after
the node runs, a router checks the state and either sends it **round again** or sends it to `END`. The test that
ends the loop is called the **exit condition**.

If the exit condition is wrong, the loop never ends. LangGraph protects you with the **recursion limit**: the most
**supersteps** a run may take before LangGraph stops it with a `GraphRecursionError`. A superstep is one round of
work in which every node that is ready runs (in this lesson, one node per round).

**Analogy:** stirring a pot "until it boils". Checking the pot after each stir is the exit condition. The kitchen
timer is the recursion limit: even if you forget to look, the timer stops you.

## Picture

The dashed arrow from `double` back to itself is the loop. The other dashed arrow is the way out.

```mermaid
flowchart LR
    S([START]) --> double
    double -. "result <= 100: go again" .-> double
    double -. "result > 100: stop" .-> E([END])
```

## Step 1: The state

`value` is the number we keep doubling, and `rounds` counts how many times `double` has run. `history` uses the
`operator.add` reducer from lesson 06, so each round adds one line.

In [1]:
import operator
from typing import Annotated, Literal, TypedDict

from langgraph.graph import StateGraph, START, END


class DoubleState(TypedDict):
    value: float
    rounds: int
    history: Annotated[list[str], operator.add]

## Step 2: The node that repeats

`double` does one round of work: double the value, add one to the round count, write a history line. It has no idea
it's in a loop. Deciding whether to repeat is the router's job.

In [2]:
def double(state: DoubleState) -> dict:
    new_value = state["value"] * 2
    return {
        "value": new_value,
        "rounds": state["rounds"] + 1,
        "history": [f"round {state['rounds'] + 1}: {state['value']:g} x 2 = {new_value:g}"],
    }

## Step 3: The router with the exit condition

`should_continue` returns `"double"` to go round again, or `END` to stop. Both go in the `Literal` hint, so LangGraph
knows both arrows exist. `END` is allowed inside `Literal` because it is just the string `'__end__'`.

In [3]:
def should_continue(state: DoubleState) -> Literal["double", END]:
    if state["value"] > 100:  # the exit condition
        return END
    return "double"


print(should_continue({"value": 50}), "|", should_continue({"value": 150}))

double | __end__


## Step 4: Build the loop

The conditional edge goes **from `double`**, and one of its destinations is `double` itself. That's the whole loop.

In [4]:
builder = StateGraph(DoubleState)
builder.add_node("double", double)
builder.add_edge(START, "double")
builder.add_conditional_edges("double", should_continue)
doubler = builder.compile()

print(doubler.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	double(double)
	__end__([<p>__end__</p>]):::last
	__start__ --> double;
	double -.-> __end__;
	double -.-> double;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Step 5: Run it

Starting at 3: 6, 12, 24, 48, 96, 192. After the sixth round the value is over 100, so the router sends the state to
`END`.

In [5]:
final = doubler.invoke({"value": 3.0, "rounds": 0})
print("value:", final["value"], "| rounds:", final["rounds"])
for line in final["history"]:
    print("  ", line)

value: 192.0 | rounds: 6
   round 1: 3 x 2 = 6
   round 2: 6 x 2 = 12
   round 3: 12 x 2 = 24
   round 4: 24 x 2 = 48
   round 5: 48 x 2 = 96
   round 6: 96 x 2 = 192


## Step 6: A second loop, a factorial step by step

The factorial of 5 is `1 × 2 × 3 × 4 × 5 = 120`. The loop multiplies `result` by a counter `i`, then adds one to `i`.
It stops once `i` has passed `n`. Same pattern as before: one node that does one step, and one router with the exit
condition.

In [6]:
class FactorialState(TypedDict):
    n: int
    i: int
    result: int
    history: Annotated[list[str], operator.add]


def multiply_step(state: FactorialState) -> dict:
    new_result = state["result"] * state["i"]
    return {
        "result": new_result,
        "i": state["i"] + 1,
        "history": [f"{state['result']} x {state['i']} = {new_result}"],
    }


def more_to_multiply(state: FactorialState) -> Literal["multiply_step", END]:
    return "multiply_step" if state["i"] <= state["n"] else END

The same wiring as the doubling loop: a fixed edge in, and a conditional edge from the node back to itself or to `END`.

In [7]:
builder = StateGraph(FactorialState)
builder.add_node("multiply_step", multiply_step)
builder.add_edge(START, "multiply_step")
builder.add_conditional_edges("multiply_step", more_to_multiply)
factorial = builder.compile()

out = factorial.invoke({"n": 5, "i": 1, "result": 1})
print("5! =", out["result"])
for line in out["history"]:
    print("  ", line)

5! = 120
   1 x 1 = 1
   1 x 2 = 2
   2 x 3 = 6
   6 x 4 = 24
   24 x 5 = 120


## Step 7: Remove the exit and hit the recursion limit

Now the mistake on purpose. A **fixed** edge from `double` back to `double` means there's no way out. LangGraph keeps
running supersteps until it reaches the recursion limit, then raises `GraphRecursionError`.

The message tells you the default limit. In the LangGraph version used here it is very high (older tutorials say 25,
but that changed), so this cell takes a few seconds while it runs about ten thousand rounds.

In [8]:
from langgraph.errors import GraphRecursionError

builder = StateGraph(DoubleState)
builder.add_node("double", double)
builder.add_edge(START, "double")
builder.add_edge("double", "double")  # no exit!
runaway = builder.compile()

try:
    runaway.invoke({"value": 3.0, "rounds": 0})
except GraphRecursionError as err:
    print("GraphRecursionError:", str(err).splitlines()[0])

GraphRecursionError: Recursion limit of 10007 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.


## Step 8: Set your own recursion limit

You don't have to wait for the default. Pass `{"recursion_limit": N}` as the second argument of `invoke`, called the
**config**. A small limit makes a runaway loop fail fast.

In [9]:
try:
    runaway.invoke({"value": 3.0, "rounds": 0}, {"recursion_limit": 50})
except GraphRecursionError as err:
    print("GraphRecursionError:", str(err).splitlines()[0])

GraphRecursionError: Recursion limit of 50 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.


The fix for the runaway graph is to put the router back. With the router, a limit of 50 is plenty, because the
correct loop finishes in about six rounds.

In [10]:
final = doubler.invoke({"value": 3.0, "rounds": 0}, {"recursion_limit": 50})
print("value:", final["value"], "| rounds:", final["rounds"])

value: 192.0 | rounds: 6


The limit counts supersteps, not just the times your node runs, and LangGraph uses a little of it for its own
bookkeeping. So leave some room. A limit of 5 is too small even for the correct loop, which needs six rounds:

In [11]:
try:
    doubler.invoke({"value": 3.0, "rounds": 0}, {"recursion_limit": 5})
except GraphRecursionError as err:
    print("GraphRecursionError:", str(err).splitlines()[0])

GraphRecursionError: Recursion limit of 5 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.


## Step 9: A safety exit in your own router

The recursion limit is a last resort: it ends the run with an error and you lose the result. Often it's nicer to add
your own limit to the exit condition, so the loop stops politely. `max_rounds` is now part of the input, and the
router stops when either test is true.

In [12]:
class SafeDoubleState(DoubleState):
    max_rounds: int


def should_continue_safe(state: SafeDoubleState) -> Literal["double", END]:
    if state["value"] > 100 or state["rounds"] >= state["max_rounds"]:
        return END
    return "double"


builder = StateGraph(SafeDoubleState)
builder.add_node("double", double)
builder.add_edge(START, "double")
builder.add_conditional_edges("double", should_continue_safe)
safe = builder.compile()

out = safe.invoke({"value": 0.001, "rounds": 0, "max_rounds": 4})
print("stopped after", out["rounds"], "rounds at value", out["value"])

stopped after 4 rounds at value 0.016


## What just happened

- **Steps 4–5** built a loop with one conditional edge from `double` back to itself. Starting at 3, it ran 6 rounds
  and stopped at `192`, the first value over 100.
- **Step 6** used the same pattern for a factorial. It printed `5! = 120` and one history line per multiplication.
- **Step 7** replaced the router with a fixed edge back to `double`. With no exit, the run stopped with
  `GraphRecursionError`, and the message showed the default limit.
- **Step 8** passed `{"recursion_limit": 50}` to fail fast. The correct loop passed easily under that limit, but
  failed under a limit of 5.
- **Step 9** added a `max_rounds` test to the router, so the loop stopped politely after 4 rounds instead of
  raising an error.

## Common mistakes

1. **No exit, or an exit that can never be true.** For example, the node forgets to return the new `value`, so the
   test `value > 100` never changes. The run ends with:
   `GraphRecursionError: Recursion limit of ... reached without hitting a stop condition.`
   Fix: check that the node changes the value the exit condition tests.
2. **Leaving `END` out of the `Literal` hint.** With `-> Literal["double"]`, returning `END` fails when the loop
   tries to stop:
   `KeyError: '__end__'`.
   Fix: list every possible destination, `Literal["double", END]`.
3. **A recursion limit that's too small for a correct loop.** The run fails with `GraphRecursionError` even though
   nothing is wrong, as with the limit of 5 in Step 8.
   Fix: pick a limit with some headroom, or add a polite safety exit like Step 9.

## Try it

1. Build a halving loop: start at `1000` and keep dividing by 2 until the value is below 1. How many rounds does it
   take, and what's the final value?
2. Without running it, predict what happens with `doubler.invoke({"value": 1e-30, "rounds": 0}, {"recursion_limit":
   50})`. Then run it. What limit would make it pass?

Answers are in [`solutions/exercises_solutions.ipynb`](../solutions/exercises_solutions.ipynb).

## Key terms

New terms from this lesson, also in [GLOSSARY.md](../GLOSSARY.md):

- **Loop**: an edge (usually conditional) that leads back to a node that already ran.
- **Exit condition**: the test in a router that sends a loop to `END`.
- **Superstep**: one round of a run, in which every node that is ready runs.
- **Recursion limit**: the most supersteps one run may take. Set it with `{"recursion_limit": N}`.
- **`GraphRecursionError`**: the error raised when a run reaches the recursion limit.
- **Config**: the second argument of `invoke`, a dict of settings for that run.